<a href="https://colab.research.google.com/github/maialist/EXSU500-Pancreatic-Segmentation/blob/main/Szirina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
import pandas as pd
import io

# Authenticate with Google.
auth.authenticate_user()
credentials, _ = google.auth.default()
drive = build("drive", "v3", credentials=credentials)

FOLDER_ID = "1HqEgzS8BV2c7xYNrZdEAnrHk7osJJ--2"
FOLDER_TYPE = "application/vnd.google-apps.folder"

def list_files_recursive(folder_id, parent_path=""):
    """List all non-trashed files inside a folder and its subfolders."""
    files = []
    page_token = None

    while True:
        response = drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()

        for item in response.get("files", []):
            path = f"{parent_path}/{item['name']}".lstrip("/")

            if item["mimeType"] == FOLDER_TYPE:
                files.extend(list_files_recursive(item["id"], path))
            else:
                files.append({**item, "path": path})

        page_token = response.get("nextPageToken")
        if not page_token:
            return files

files = list_files_recursive(FOLDER_ID)
df = pd.DataFrame(files, columns=["id", "name", "mimeType", "path"])

print(f"Found {len(files)} files.")
display(df)

def read_file(file_id, export_mime_type=None):
    """Return file contents as bytes.

    For native Google Docs/Sheets/Slides, supply an export MIME type.
    """
    if export_mime_type:
        return drive.files().export_media(
            fileId=file_id,
            mimeType=export_mime_type,
        ).execute()

    return drive.files().get_media(
        fileId=file_id,
        supportsAllDrives=True,
    ).execute()

Found 11 files.


,id,name,mimeType,path
0,1qVrpV7vmhIsUxFiH189LmAn0ALbAPrgS,Task08_HepaticVessel.tar,application/x-tar,Task08_HepaticVessel.tar
1,1YZQFSonulXuagMIfbJkZeTFJ6qEUuUxL,Task07_Pancreas.tar,application/x-tar,Task07_Pancreas.tar
2,1jyVGUGyxKBXV6_9ivuZapQS8eUJXCIpu,Task03_Liver.tar,application/x-tar,Task03_Liver.tar
3,1A2IU8Sgea1h3fYLpYtFb2v7NYdMjvEhU,Task01_BrainTumour.tar,application/x-tar,Task01_BrainTumour.tar
4,1m7tMpE9qEcQGQjL_BdMD-Mvgmc44hG1Y,Task10_Colon.tar,application/x-tar,Task10_Colon.tar
5,1jzeNU1EKnK81PyTsrx0ujfNl-t0Jo8uE,Task09_Spleen.tar,application/x-tar,Task09_Spleen.tar
6,1RzPB1_bqzQhlWvU-YGvZzhx2omcDh38C,Task04_Hippocampus.tar,application/x-tar,Task04_Hippocampus.tar
7,1I1LR7XjyEZ-VBQ-Xruh31V7xExMjlVvi,Task06_Lung.tar,application/x-tar,Task06_Lung.tar
8,1Ff7c21UksxyT4JfETjaarmuKEjdqe1-a,Task05_Prostate.tar,application/x-tar,Task05_Prostate.tar
9,1wEB2I6S6tQBVEPxir8cA5kFB8gTQadYY,Task02_Heart.tar,application/x-tar,Task02_Heart.tar


In [3]:
# Example: load the first CSV into pandas.
csv_files = df[df["name"].str.lower().str.endswith(".csv")]

if not csv_files.empty:
    file_id = csv_files.iloc[0]["id"]
    data = pd.read_csv(io.BytesIO(read_file(file_id)))
    display(data.head())

# Other examples:
# text = read_file("FILE_ID").decode("utf-8")
# doc_text = read_file("GOOGLE_DOC_ID", "text/plain").decode("utf-8")
# sheet_bytes = read_file(
#     "GOOGLE_SHEET_ID",
#     "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
# )

In [ ]:
import os
import tarfile
import tempfile
import urllib.request
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from PIL import Image

archive_path = "/content/Task07_Pancreas.tar"
target_filename = "Task07_Pancreas.tar"

# 1. Download Task07_Pancreas.tar from the official MSD repository mirror
if not os.path.exists(archive_path):
    # Direct mirror link for MSD Pancreas dataset
    decathlon_url = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar"

    print(f"Downloading {target_filename} directly to Colab from the MSD mirror...")
    try:
        def report_progress(block_num, block_size, total_size):
            read_so_far = block_num * block_size
            if total_size > 0:
                percent = read_so_far * 1e2 / total_size
                print(f"\rProgress: {percent:.1f}% ({read_so_far / (1024*1024):.1f} MB of {total_size / (1024*1024):.1f} MB)", end="", flush=True)
            else:
                print(f"\rDownloaded: {read_so_far / (1024*1024):.1f} MB", end="", flush=True)

        urllib.request.urlretrieve(decathlon_url, archive_path, reporthook=report_progress)
        print("\nDownload complete.")
    except Exception as download_error:
        if os.path.exists(archive_path):
            os.remove(archive_path)
        print("\nMirror download failed. The repository may be temporarily offline.")
        raise download_error
else:
    print(f"Local copy of {archive_path} detected. Skipping download.")

# 2. Preview the Images/Scans
COLS = 4
BATCH_SIZE = 12
batch = []
previewed = 0

def show_batch(items):
    rows = (len(items) + COLS - 1) // COLS
    fig, axes = plt.subplots(rows, COLS, figsize=(16, rows * 4), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (name, image, medical) in zip(axes.flat, items):
        if medical:
            finite = image[np.isfinite(image)]
            low, high = np.percentile(finite, [1, 99]) if finite.size else (0, 1)
            ax.imshow(image, cmap="gray", vmin=low, vmax=high if high > low else low + 1)
        else:
            ax.imshow(image)
        ax.set_title(name, fontsize=8, wrap=True)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

if os.path.exists(archive_path):
    print(f"Extracting and previewing {archive_path}...")
    with tarfile.open(archive_path, "r:*") as archive:
        for member in archive:
            if not member.isfile():
                continue
            name = member.name
            lower = name.lower()
            if os.path.basename(name).startswith("._"):
                continue

            medical = lower.endswith((".nii", ".nii.gz"))
            ordinary = lower.endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"))
            if not (medical or ordinary):
                continue

            try:
                with archive.extractfile(member) as source:
                    if medical:
                        suffix = ".nii.gz" if lower.endswith(".gz") else ".nii"
                        with tempfile.TemporaryDirectory() as tmp:
                            scan_path = os.path.join(tmp, "scan" + suffix)
                            with open(scan_path, "wb") as target:
                                import shutil
                                shutil.copyfileobj(source, target)
                            scan = nib.as_closest_canonical(nib.load(scan_path))
                            index = (slice(None), slice(None), scan.shape[2] // 2) + (0,) * (len(scan.shape) - 3)
                            image = np.rot90(np.array(scan.dataobj[index], dtype=np.float32))
                    else:
                        with Image.open(source) as picture:
                            picture.thumbnail((600, 600))
                            image = np.array(picture.convert("RGB"))

                batch.append((name, image, medical))
                previewed += 1
                if len(batch) == BATCH_SIZE:
                    show_batch(batch)
                    batch = []
            except Exception as error:
                print(f"Could not preview {name}: {error}")

    if batch:
        show_batch(batch)
    print(f"Previewed {previewed} images/scans from {target_filename}.")

Progress: 16.3% (1908.3 MB of 11720.6 MB)

In [7]:
import os

# Path to the potentially truncated archive
corrupted_path = "/content/Task07_Pancreas.tar"

if os.path.exists(corrupted_path):
    size_mb = os.path.getsize(corrupted_path) / (1024 * 1024)
    print(f"The current local file size is: {size_mb:.2f} MB")

    # Delete the partial file so we can perform a clean download once the quota resets
    os.remove(corrupted_path)
    print(f"Deleted corrupted file at '{corrupted_path}'. Run the previous cell again to attempt a fresh download (if your Google Drive quota has reset).")
else:
    print("No local file found to delete.")

The current local file size is: 1472.00 MB
Deleted corrupted file at '/content/Task07_Pancreas.tar'. Run the previous cell again to attempt a fresh download (if your Google Drive quota has reset).


### Batch Preview Local Archives

Upload any of your manually downloaded `.tar` files (e.g., `Task07_Pancreas.tar`, `Task03_Liver.tar`, etc.) directly to the `/content/` folder in your left-hand side panel.

Then, run this cell to automatically scan for any uploaded archives and preview their slices.

In [9]:
import os
import glob
import tarfile
import tempfile
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from PIL import Image

# Find all uploaded tar files in /content/
local_tars = glob.glob("/content/*.tar")

COLS = 4
BATCH_SIZE = 12

def show_batch(items, archive_name):
    rows = (len(items) + COLS - 1) // COLS
    fig, axes = plt.subplots(
        rows, COLS,
        figsize=(16, rows * 4),
        squeeze=False,
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (name, image, medical) in zip(axes.flat, items):
        if medical:
            finite = image[np.isfinite(image)]
            low, high = (
                np.percentile(finite, [1, 99])
                if finite.size else (0, 1)
            )
            ax.imshow(
                image, cmap="gray",
                vmin=low,
                vmax=high if high > low else low + 1,
            )
        else:
            ax.imshow(image)

        ax.set_title(name, fontsize=8, wrap=True)

    plt.suptitle(f"Previews from {archive_name}", fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

if not local_tars:
    print("No local .tar files detected in /content/. Please upload your downloaded archives first!")
else:
    for archive_path in local_tars:
        archive_name = os.path.basename(archive_path)
        print(f"\nProcessing uploaded archive: {archive_name}...")

        batch = []
        previewed = 0

        try:
            with tarfile.open(archive_path, "r:*") as archive:
                for member in archive:
                    if not member.isfile():
                        continue

                    name = member.name
                    lower = name.lower()

                    if os.path.basename(name).startswith("._"):
                        continue

                    medical = lower.endswith((".nii", ".nii.gz"))
                    ordinary = lower.endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"))

                    if not (medical or ordinary):
                        continue

                    try:
                        with archive.extractfile(member) as source:
                            if medical:
                                suffix = ".nii.gz" if lower.endswith(".gz") else ".nii"
                                with tempfile.TemporaryDirectory() as tmp:
                                    scan_path = os.path.join(tmp, "scan" + suffix)
                                    with open(scan_path, "wb") as target:
                                        import shutil
                                        shutil.copyfileobj(source, target)

                                    scan = nib.as_closest_canonical(nib.load(scan_path))
                                    index = (
                                        (slice(None), slice(None), scan.shape[2] // 2)
                                        + (0,) * (len(scan.shape) - 3)
                                    )
                                    image = np.rot90(
                                        np.array(scan.dataobj[index], dtype=np.float32)
                                    )
                            else:
                                with Image.open(source) as picture:
                                    picture.thumbnail((600, 600))
                                    image = np.array(picture.convert("RGB"))

                        batch.append((name, image, medical))
                        previewed += 1

                        if len(batch) == BATCH_SIZE:
                            show_batch(batch, archive_name)
                            batch = []

                    except Exception as error:
                        print(f"Could not preview {name}: {error}")

            if batch:
                show_batch(batch, archive_name)
            print(f"Finished {archive_name}: Previewed {previewed} items.")

        except Exception as archive_error:
            print(f"Error reading {archive_name}: {archive_error}. The file may be corrupted or incomplete.")

No local .tar files detected in /content/. Please upload your downloaded archives first!
